In [ ]:
import google.colab
google.colab.drive.mount("drive/")

In [ ]:
pip uninstall -y tensorflow

Found existing installation: tensorflow 2.17.1
Uninstalling tensorflow-2.17.1:
  Successfully uninstalled tensorflow-2.17.1


In [ ]:
pip install tensorflow-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.0/230.0 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 28.2 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.17.1
    Uninstalling tensorboard-2.17.1:
      Successfully uninstalled tensorboard-2.17.1


In [1]:
!pip install "tf-models-official==2.13.*"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of tf-keras to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.7/479.7 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import numpy as np
import pandas as pd
import time
import datetime
import gc
import random
from nltk.corpus import stopwords
import re

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler,random_split
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import transformers
from transformers import BertForSequenceClassification, AdamW, BertConfig,BertTokenizer,get_linear_schedule_with_warmup

c:\Users\Hasnain Naqvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [65]:
# df = pd.read_csv("/content/drive/MyDrive/combined.csv")
df = pd.read_csv("../../dataset/cleaned_data/cleaned_df_stopword.csv")


In [ ]:
df.columns

Index(['Unnamed: 0', 'Author_name', 'topic', 'title', 'Abstract', 'year',
       'Source', 'label'],
      dtype='object')

In [ ]:
df = df.drop("Unnamed: 0", axis=1)
df.head()

,Author_name,topic,title,Abstract,year,Source,label
0,"Iain Carmichael, J. S. Marron",machine learning,Data Science vs. Statistics: Two Cultures?,Data science is the business of learning from ...,2017,arxiv,human
1,"Jannis Kueck, Ye Luo, Martin Spindler, Zigan Wang",machine learning,Estimation and Inference of Treatment Effects ...,Empirical researchers are increasingly faced w...,2017,arxiv,human
2,Jason Toy,machine learning,SenseNet: 3D Objects Database and Tactile Simu...,The majority of artificial intelligence resear...,2017,arxiv,human
3,"Yu-Ren Liu, Yi-Qi Hu, Hong Qian, Chao Qian, Ya...",machine learning,ZOOpt: Toolbox for Derivative-Free Optimization,Recent advances in derivative-free optimizatio...,2017,arxiv,human
4,Abien Fred Agarap,machine learning,Towards Building an Intelligent Anti-Malware S...,Effective and efficient mitigation of malware ...,2017,arxiv,human


In [ ]:
print(df.isnull().sum())

Author_name    0
topic          0
title          0
Abstract       4
year           0
Source         0
label          0
dtype: int64


In [ ]:
print(df[df.isnull().any(axis=1)])


       Unnamed: 0 Author_name                    topic  \
30786        1956       llama  Artificial Intelligence   
30869        2039       llama       Internet of Things   
30902        2072       llama       Internet of Things   
31036        2206       llama       Internet of Things   

                                                   title Abstract  year  \
30786                    Implementing the Deep Q-Network      NaN  2024   
30869  Learning-Based Computation Offloading for IoT ...      NaN  2024   
30902  Memory-based Combination PUFs for Device Authe...      NaN  2024   
31036                  Correctness of the Chord Protocol      NaN  2024   

      Source label  
30786  llama    AI  
30869  llama    AI  
30902  llama    AI  
31036  llama    AI  


In [3]:
df_cleaned = df.dropna(subset=['Abstract'])


In [4]:
print(df_cleaned[df_cleaned.isnull().any(axis=1)])


Empty DataFrame
Columns: [Unnamed: 0, Author_name, topic, title, Abstract, year, Source, label]
Index: []


In [ ]:
print(df_cleaned.isnull().sum())

Author_name    0
topic          0
title          0
Abstract       0
year           0
Source         0
label          0
dtype: int64


In [7]:
df_cleaned["Abstract"].values[:5]

,Unnamed: 0,Author_name,topic,title,Abstract,year,Source,label
0,0,"Iain Carmichael, J. S. Marron",machine learning,Data Science vs. Statistics: Two Cultures?,Data science is the business of learning from ...,2017,arxiv,human
1,1,"Jannis Kueck, Ye Luo, Martin Spindler, Zigan Wang",machine learning,Estimation and Inference of Treatment Effects ...,Empirical researchers are increasingly faced w...,2017,arxiv,human
2,2,Jason Toy,machine learning,SenseNet: 3D Objects Database and Tactile Simu...,The majority of artificial intelligence resear...,2017,arxiv,human
3,3,"Yu-Ren Liu, Yi-Qi Hu, Hong Qian, Chao Qian, Ya...",machine learning,ZOOpt: Toolbox for Derivative-Free Optimization,Recent advances in derivative-free optimizatio...,2017,arxiv,human
4,4,Abien Fred Agarap,machine learning,Towards Building an Intelligent Anti-Malware S...,Effective and efficient mitigation of malware ...,2017,arxiv,human
...,...,...,...,...,...,...,...,...
43293,14463,Llama3,Social Influence,Influence of adaptive mesh refinement and the ...,This study investigate the effects of adaptive...,2024,Llama3,AI
43294,14464,Llama3,Social Influence,Random trees constructed by aggregation,This study introduces a new approach to constr...,2024,Llama3,AI
43295,14465,Llama3,Social Influence,Colliding Filaments and a Massive Dense Core i...,This study presents a high-resolution analysis...,2024,Llama3,AI
43296,14466,Llama3,Social Influence,Time-optimal navigation through quantum wind,This research paper presents a novel approach ...,2024,Llama3,AI


In [8]:
df_cleaned = df_cleaned.drop("Unnamed: 0", axis=1)

In [9]:
# Count the number of words in each abstract
df_cleaned["Word_Count"] = df_cleaned["Abstract"].apply(lambda x: len(str(x).split()) if pd.notnull(x) else 0)


In [10]:
# Filter rows where the word count is greater than 350
filtered_df = df_cleaned[df_cleaned["Word_Count"] > 350]

In [11]:
filtered_df.to_csv("filtered_df.csv")

In [ ]:
# df_cleaned = df_cleaned.drop("Unnamed: 0", axis=1)

In [12]:
df_cleaned = df_cleaned[df_cleaned['Word_Count'] <= 350]

In [13]:
df_cleaned.to_csv("cleaned_df.csv")

In [14]:
print(len(df_cleaned))
print(len(df))

43264
43298


## STOPWORD APPROACH

In [78]:
import nltk
nltk.download('stopwords')
sw = stopwords.words('english')

[nltk_data] Downloading package stopwords to C:\Users\Hasnain
[nltk_data]     Naqvi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [79]:


def clean_text(text):
    try:

        text = text.lower()
    except Exception as e:
        print(text)
        print(e)
        return

    text = re.sub(r"[^a-zA-Z?.!,¿]+", " ", text) # replacing everything with space except (a-z, A-Z, ".", "?", "!", ",")

    punctuations = '@#!?+&*[]-%.:/();$=><|{}^' + "'`" + '_'
    for p in punctuations:
        text = text.replace(p,'') #Removing punctuations

    text = [word.lower() for word in text.split() if word.lower() not in sw]

    text = " ".join(text) #removing stopwords


    return text


In [17]:
df_cleaned["Abstract"] = df_cleaned['Abstract'].apply(lambda x: clean_text(x))
df_cleaned.to_csv("cleaned_df_stopword.csv")


In [18]:

abstracts = df_cleaned.Abstract.values
labels = df_cleaned.label.values


In [23]:

# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

c:\Users\Hasnain Naqvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [21]:
abstracts[0]

'data science business learning data, traditionally business statistics data science, however, often understood broader, task driven computationally oriented version statistics term data science broader idea conveys origins statistics reaction narrower view data analysis expanding upon views number statisticians, paper encourages big tent view data analysis examine evolving approaches modern data analysis relate existing discipline statistics eg exploratory analysis, machine learning, reproducibility, computation, communication role theory finally, discuss trends mean future statistics highlighting promising directions communication, education research'

In [20]:

print(' Original: ', abstracts[0])

# Print the sentence split into tokens.
print('Tokenized: ', tokenizer.tokenize(abstracts[0]))

# Print the sentence mapped to token ids.
print('Token IDs: ', tokenizer.convert_tokens_to_ids(tokenizer.tokenize(abstracts[0])))

 Original:  data science business learning data, traditionally business statistics data science, however, often understood broader, task driven computationally oriented version statistics term data science broader idea conveys origins statistics reaction narrower view data analysis expanding upon views number statisticians, paper encourages big tent view data analysis examine evolving approaches modern data analysis relate existing discipline statistics eg exploratory analysis, machine learning, reproducibility, computation, communication role theory finally, discuss trends mean future statistics highlighting promising directions communication, education research
Tokenized:  ['data', 'science', 'business', 'learning', 'data', ',', 'traditionally', 'business', 'statistics', 'data', 'science', ',', 'however', ',', 'often', 'understood', 'broader', ',', 'task', 'driven', 'computational', '##ly', 'oriented', 'version', 'statistics', 'term', 'data', 'science', 'broader', 'idea', 'convey', '

In [ ]:

max_len = 0

for abstract in abstracts:

    # Tokenize the text and add `[CLS]` and `[SEP]` tokens.
    input_ids = tokenizer.encode(abstract, add_special_tokens=True)

    # Update the maximum sentence length.
    max_len = max(max_len, len(input_ids))

print('Max abstract length: ', max_len)

Max abstract length:  484


In [22]:
input_ids = []
attention_masks = []

# For every abstratc
for abstract in abstracts:
    # `encode_plus` will:
    #   (1) Tokenize the sentence.
    #   (2) Prepend the `[CLS]` token to the start.
    #   (3) Append the `[SEP]` token to the end.
    #   (4) Map tokens to their IDs.
    #   (5) Pad or truncate the sentence to `max_length`
    #   (6) Create attention masks for [PAD] tokens.
    encoded_dict = tokenizer.encode_plus(
                        abstract                  ,    # Sentence to encode.
                        add_special_tokens = True ,    # Add '[CLS]' and '[SEP]'
                        max_length = 512          ,    # Pad & truncate all sentences.
                        pad_to_max_length = True,
                        return_attention_mask = True,  # Construct attn. masks.
                        return_tensors = 'pt',         # Return pytorch tensors.
                   )

    # Add the encoded sentence to the list.
    input_ids.append(encoded_dict['input_ids'])

    # And its attention mask (simply differentiates padding from non-padding).
    attention_masks.append(encoded_dict['attention_mask'])

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)


Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:2673: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


In [23]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df_cleaned['label_encoded'] = le.fit_transform(df_cleaned['label'])
labels = df_cleaned.label_encoded.values
labels = torch.tensor(labels)

In [8]:
import torch

print("Tensors saved to 'encoded_data.pt'")


Tensors saved to 'encoded_data.pt'


# reload tensors

In [84]:
encoded_data = torch.load('tensors/encoded_data.pt')



C:\Users\Hasnain Naqvi\AppData\Local\Temp\ipykernel_7632\4077019788.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  encoded_data = torch.load('tensors/encoded_data.pt')


In [85]:

input_ids = encoded_data['input_ids']
attention_masks = encoded_data['attention_masks']
labels = encoded_data['labels']

print("Tensors loaded successfully!")

Tensors loaded successfully!


In [6]:
attention_masks.shape

torch.Size([43264, 512])

In [13]:

# print('Original: ', abstracts[0])
# print('Token IDs:', input_ids[0])
print("label: ", labels[0])

label:  tensor(1)


In [91]:
dataset = TensorDataset(input_ids, attention_masks, labels)

# Create a 75-15-10 train-validation-testing split.
# Calculate the number of samples to include in each set.
train_size = int(0.75 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = int(0.1 * len(dataset)) + 1 # due to conversion to integer


In [92]:

# Divide the dataset by randomly selecting samples.
# train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])


print('{:>5,} training samples'.format(train_size))
print('{:>5,} validation samples'.format(val_size))
print('{:>5,} testing samples'.format(test_size))

32,448 training samples
6,489 validation samples
4,327 testing samples


In [93]:
# The DataLoader needs to know our batch size for training, so we specify it
# here. For fine-tuning BERT on a specific task, the authors recommend a batch
# size of 16 or 32.
batch_size = 16

# Create the DataLoaders for our training and validation sets.
# We'll take training samples in random order.
train_dataloader = DataLoader(
            train_dataset,  # The training samples.
            sampler = RandomSampler(train_dataset), # Select batches randomly
            batch_size = batch_size # Trains with this batch size.
        )

# For validation the order doesn't matter, so we'll just read them sequentially.
validation_dataloader = DataLoader(
            val_dataset, # The validation samples.
            sampler = RandomSampler(val_dataset), # Pull out batches sequentially.
            batch_size = batch_size # Evaluate with this batch size.
        )

# For testing the order doesn't matter, so we'll just read them sequentially.
testing_dataloader = DataLoader(
            test_dataset,      # The validation samples.
            sampler = RandomSampler(test_dataset), # Pull out batches sequentially.
            batch_size = batch_size # Evaluate with this batch size.
        )

In [17]:
# Load BertForSequenceClassification, the pretrained BERT model with a single
# linear classification layer on top.
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", # Use the 12-layer BERT model, with an uncased vocab.
    num_labels = 2, # The number of output labels--2 for binary classification.
                    # You can increase this for multi-class tasks.
    output_attentions = True, # Whether the model returns attentions weights.
    output_hidden_states = True, # Whether the model returns all hidden-states.
)



/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
# if device == "cuda:0":
# # Tell pytorch to run this model on the GPU.
#     model = model.cuda()
model = model.to("cuda")

In [19]:
optimizer = AdamW(model.parameters(),
                  lr = 2e-5, # args.learning_rate - default is 5e-5, our notebook had 2e-5
                  eps = 1e-8 # args.adam_epsilon  - default is 1e-8.
                )

/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


In [20]:
# Number of training epochs. The BERT authors recommend between 2 and 4.
# We chose to run for 4, but we'll see later that this may be over-fitting the
# training data.
epochs = 4

# Total number of training steps is [number of batches] x [number of epochs].
# (Note that this is not the same as the number of training samples).
total_steps = len(train_dataloader) * epochs
total_steps


8112

In [21]:

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

In [22]:
# Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

In [23]:
def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))

In [24]:
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)
training_stats = []
device = "cuda"
# Measure the total training time for the whole run.

In [25]:
total_t0 = time.time()

# For each epoch...
for epoch_i in range(0, epochs):

    # ========================================
    #               Training
    # ========================================
    # Perform one full pass over the training set.
    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    print('Training...')
    # Measure how long the training epoch takes.
    t0 = time.time()
    total_train_loss = 0
    model.train()
    for step, batch in enumerate(train_dataloader):
        # Unpack this training batch from our dataloader.
        #
        #  As we unpack the batch, we'll also copy each tensor to the device using the
        # `to` method.
        #
        # `batch` contains three pytorch tensors:
        #   [0]: input ids
        #   [1]: attention masks
        #   [2]: labels
        print(step) if step % 40 == 0 else None
        b_input_ids  = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels     = batch[2].to(device)
        optimizer.zero_grad()
        output = model(b_input_ids,
                             token_type_ids=None,
                             attention_mask=b_input_mask,
                             labels=b_labels)
        loss = output.loss
        total_train_loss += loss.item()
        # Perform a backward pass to calculate the gradients.
        loss.backward()
        # Clip the norm of the gradients to 1.0.
        # This is to help prevent the "exploding gradients" problem.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        # Update parameters and take a step using the computed gradient.
        # The optimizer dictates the "update rule"--how the parameters are
        # modified based on their gradients, the learning rate, etc.
        optimizer.step()
        # Update the learning rate.
        scheduler.step()

    # Calculate the average loss over all of the batches.
    avg_train_loss = total_train_loss / len(train_dataloader)

    # Measure how long this epoch took.
    training_time = format_time(time.time() - t0)
    print("")
    print("  Average training loss: {0:.2f}".format(avg_train_loss))
    print("  Training epoch took: {:}".format(training_time))
    # ========================================
    #               Validation
    # ========================================
    # After the completion of each training epoch, measure our performance on
    # our validation set.
    print("")
    print("Running Validation...")
    t0 = time.time()
    # Put the model in evaluation mode--the dropout layers behave differently
    # during evaluation.
    model.eval()
    # Tracking variables
    total_eval_accuracy = 0
    best_eval_accuracy = 0
    total_eval_loss = 0
    nb_eval_steps = 0
    # Evaluate data for one epoch
    for batch in validation_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        # Tell pytorch not to bother with constructing the compute graph during
        # the forward pass, since this is only needed for backprop (training).
        with torch.no_grad():
            output= model(b_input_ids,
                                   token_type_ids=None,
                                   attention_mask=b_input_mask,
                                   labels=b_labels)
        loss = output.loss
        total_eval_loss += loss.item()
        # Move logits and labels to CPU if we are using GPU
        logits = output.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()
        # Calculate the accuracy for this batch of test sentences, and
        # accumulate it over all batches.
        total_eval_accuracy += flat_accuracy(logits, label_ids)
    # Report the final accuracy for this validation run.
    avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
    print("  Accuracy: {0:.2f}".format(avg_val_accuracy))
    # Calculate the average loss over all of the batches.
    avg_val_loss = total_eval_loss / len(validation_dataloader)
    # Measure how long the validation run took.
    validation_time = format_time(time.time() - t0)
    if avg_val_accuracy > best_eval_accuracy:
        torch.save(model, 'bert_model')
        best_eval_accuracy = avg_val_accuracy
    #print("  Validation Loss: {0:.2f}".format(avg_val_loss))
    #print("  Validation took: {:}".format(validation_time))
    # Record all statistics from this epoch.
    training_stats.append(
        {
            'epoch': epoch_i + 1,
            'Training Loss': avg_train_loss,
            'Valid. Loss': avg_val_loss,
            'Valid. Accur.': avg_val_accuracy,
            'Training Time': training_time,
            'Validation Time': validation_time
        }
    )
print("")
print("Training complete!")

print("Total training took {:} (h:mm:ss)".format(format_time(time.time()-total_t0)))


======== Epoch 1 / 4 ========
Training...
0


BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


40
80
120
160
200
240
280
320
360
400
440
480
520
560
600
640
680
720
760
800
840
880
920
960
1000
1040
1080
1120
1160
1200
1240
1280
1320
1360
1400
1440
1480
1520
1560
1600
1640
1680
1720
1760
1800
1840
1880
1920
1960
2000

  Average training loss: 0.22
  Training epoch took: 0:48:00

Running Validation...
  Accuracy: 0.94

======== Epoch 2 / 4 ========
Training...
0
40
80
120
160
200
240
280
320
360
400
440
480
520
560
600
640
680
720
760
800
840
880
920
960
1000
1040
1080
1120
1160
1200
1240
1280
1320
1360
1400
1440
1480
1520
1560
1600
1640
1680
1720
1760
1800
1840
1880
1920
1960
2000

  Average training loss: 0.12
  Training epoch took: 0:48:01

Running Validation...
  Accuracy: 0.93

======== Epoch 3 / 4 ========
Training...
0
40


KeyboardInterrupt: 

# reload model


In [ ]:
# Load the saved model
model = torch.load('bert_model')

# Put the model back in training mode
model.train()


NameError: name 'torch' is not defined

In [ ]:
for epoch_i in range(2, 5):

    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, 2 * epochs))
    print('Training...')
    t0 = time.time()
    total_train_loss = 0

    for step, batch in enumerate(train_dataloader):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        optimizer.zero_grad()
        output = model(b_input_ids,
                       token_type_ids=None,
                       attention_mask=b_input_mask,
                       labels=b_labels)
        loss = output.loss
        total_train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_dataloader)
    training_time = format_time(time.time() - t0)
    print("")
    print("  Average training loss: {0:.2f}".format(avg_train_loss))
    print("  Training epoch took: {:}".format(training_time))

    print("")
    print("Running Validation...")
    t0 = time.time()
    model.eval()
    total_eval_accuracy = 0
    total_eval_loss = 0
    nb_eval_steps = 0

    for batch in validation_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        with torch.no_grad():
            output = model(b_input_ids,
                           token_type_ids=None,
                           attention_mask=b_input_mask,
                           labels=b_labels)
        loss = output.loss
        total_eval_loss += loss.item()

        logits = output.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()
        total_eval_accuracy += flat_accuracy(logits, label_ids)

    avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
    avg_val_loss = total_eval_loss / len(validation_dataloader)
    validation_time = format_time(time.time() - t0)

    print("  Accuracy: {0:.2f}".format(avg_val_accuracy))
    if avg_val_accuracy > best_eval_accuracy:
        torch.save(model, 'bert_model')
        best_eval_accuracy = avg_val_accuracy

    training_stats.append(
        {
            'epoch': epoch_i + 1,
            'Training Loss': avg_train_loss,
            'Valid. Loss': avg_val_loss,
            'Valid. Accur.': avg_val_accuracy,
            'Training Time': training_time,
            'Validation Time': validation_time
        }
    )
print("")
print("Training complete!")


# TESTING DATASET

In [28]:
import numpy as np
from sklearn.metrics import accuracy_score

predictions = []
true_labels = []  # To store ground truth labels

for batch in testing_dataloader:
    b_input_ids = batch[0].to(device)
    b_input_mask = batch[1].to(device)
    b_labels = batch[2].to(device)  # Assuming labels are at index 2

    with torch.no_grad():
        outputs = model(b_input_ids, attention_mask=b_input_mask)
        logits = outputs.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.cpu().numpy()

        # Store predictions and true labels
        pred_flat = np.argmax(logits, axis=1).flatten()
        predictions.extend(pred_flat)
        true_labels.extend(label_ids)

# Calculate accuracy
accuracy = accuracy_score(true_labels, predictions)
print(f"Accuracy: {accuracy * 100:.2f}%")


In [32]:
true_labels[:5]

array([0, 0, 1, 0, 0])

In [33]:
predictions[:5]

array([0, 0, 1, 1, 0])

C:\Users\Hasnain Naqvi\AppData\Local\Temp\ipykernel_7632\790308016.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load('model/bert_model', map_location=de

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [34]:
import numpy as np
from sklearn.metrics import accuracy_score

predictions = []
true_labels = []  # To store ground truth labels

for batch in testing_dataloader:
    b_input_ids = batch[0].to(device)
    b_input_mask = batch[1].to(device)
    b_labels = batch[2].to(device)  # Assuming labels are at index 2

    with torch.no_grad():
        outputs = model(b_input_ids, attention_mask=b_input_mask)
        logits = outputs.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.cpu().numpy()

        # Store predictions and true labels
        pred_flat = np.argmax(logits, axis=1).flatten()
        predictions.extend(pred_flat)
        true_labels.extend(label_ids)

# Calculate accuracy
accuracy = accuracy_score(true_labels, predictions)
print(f"Accuracy: {accuracy * 100:.2f}%")


Accuracy: 94.48%


# FOR SINGLE ABSTRACT 


In [72]:
single_abstract = df["Abstract"].values[40000]
label = df["label"].values[40000]

In [73]:
label

'AI'

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

In [87]:
text = clean_text('''Large language models (LLMs) excel at solving complex math problems by dividing them into smaller subproblems. However, their reasoning within these subproblems often falters due to a mismatch between the granularity of in-context learning (ICL) examples and the fine-grained reasoning steps required. To address this, we introduce BoostStep, a novel method that aligns example granularity with reasoning steps and provides highly relevant examples using a "first-try" strategy. BoostStep significantly improves the reasoning quality within each step, enhancing the performance of leading LLMs like GPT-4 and Qwen2.5-Math-72B on various mathematical benchmarks. When combined with Monte Carlo Tree Search (MCTS) methods, BoostStep further refines both candidate solution generation and decision-making, demonstrating substantial gains in overall problem-solving performance.''')

In [88]:
encoded_dict = tokenizer.encode_plus(
                    text                  ,    # Sentence to encode.
                    add_special_tokens = True ,    # Add '[CLS]' and '[SEP]'
                    max_length = 512          ,    # Pad & truncate all sentences.
                    pad_to_max_length = True,
                    return_attention_mask = True,  # Construct attn. masks.
                    return_tensors = 'pt',         # Return pytorch tensors.
                )
# # Convert the lists into tensors.
# input_ids = torch.cat(encoded_dict["input_ids"], dim=1)
# attention_masks = torch.cat(encoded_dict["attention_masks"], dim=1)

c:\Users\Hasnain Naqvi\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:2870: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


In [30]:
from transformers import BertForSequenceClassification

# Load the saved model
# Specify to load the model onto the CPU
device = torch.device("cpu")
model = torch.load('model/bert_model', map_location=device)
# model.load_state_dict(torch.load('bert_model'))
model.eval()


C:\Users\Hasnain Naqvi\AppData\Local\Temp\ipykernel_7632\790308016.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load('model/bert_model', map_location=de

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [89]:
with torch.no_grad():
    outputs = model(encoded_dict["input_ids"], attention_mask=encoded_dict["attention_mask"])
    logits = outputs.logits
    print(logits)
    logits = logits.detach().cpu().numpy()
    print(logits)

    # Store predictions and true labels
    pred_flat = np.argmax(logits, axis=1).flatten()


tensor([[ 3.4459, -3.5453]])
[[ 3.4458756 -3.5452724]]


In [90]:
np.argmax(logits, axis=1)

array([0], dtype=int64)

In [64]:
pred_flat

array([0], dtype=int64)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

predictions = []
true_labels = []  # To store ground truth labels

for batch in testing_dataloader:
    b_input_ids = batch[0].to(device)
    b_input_mask = batch[1].to(device)
    b_labels = batch[2].to(device)  # Assuming labels are at index 2

    with torch.no_grad():
        outputs = model(b_input_ids, attention_mask=b_input_mask)
        logits = outputs.logits
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.cpu().numpy()

        # Store predictions and true labels
        pred_flat = np.argmax(logits, axis=1).flatten()
        predictions.extend(pred_flat)
        true_labels.extend(label_ids)

# Calculate accuracy
accuracy = accuracy_score(true_labels, predictions)
print(f"Accuracy: {accuracy * 100:.2f}%")
